# 04 Strict Timeline & Referee Validation — Workstream 4

Owner: Anas

## This notebook
1. Implements the **strict b=0 exponential decline challenger** with physical water caps
2. Generates backtest predictions across all 4 experiments
3. Runs them through the **referee backtest_engine.py** for standardized scoring
4. Produces deployable 2008→2028 conditional forecasts
5. Exports the W4 handoff package

## Challenger definition
- **Oil**: strict exponential (b=0), anchored at cutoff terminal rate
- **Gas**: nonnegative produced-GOR rule from historical median
- **Water**: relaxation dynamics with optional rate cap (2x observed 2008)
- **Integration**: daily trapezoid from cutoff, query-grid invariant

In [1]:
%pip install numpy pandas pyarrow scipy matplotlib seaborn -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys, importlib, warnings, hashlib, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import least_squares
from scipy.integrate import cumulative_trapezoid
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

_curr = Path.cwd().resolve()
REPO_ROOT = None
temp = _curr
while temp != temp.parent:
    if (temp / "Anas").exists() and (temp / "data").exists():
        REPO_ROOT = temp
        break
    temp = temp.parent

if REPO_ROOT is not None:
    ROOT = REPO_ROOT / "Anas"
    SHARED = REPO_ROOT / "data" / "shared"
else:
    ROOT = Path("/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/mwap-exdw-surrogate-model/Anas-db")
    SHARED = Path("/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/data")
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "strict_challenger"
for d in [OUT / "figures", OUT / "tables", OUT / "models", OUT / "data"]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
import temporal_utils as tu
importlib.reload(tu)
import backtest_engine as be
importlib.reload(be)

tu_hash = hashlib.md5((ROOT / "temporal_utils.py").read_bytes()).hexdigest()[:8]
be_hash = hashlib.md5((ROOT / "backtest_engine.py").read_bytes()).hexdigest()[:8]
print(f"temporal_utils.py hash: {tu_hash}")
print(f"backtest_engine.py hash: {be_hash}")

unc, curves = tu.load_shared(SHARED)
unc_dev = unc.loc[1:85]
targets = pd.read_parquet(DATA / "temporal_targets.parquet")

SEED = 42
YEAR_DAYS = 365.25
CUTOFF = pd.Timestamp("2008-01-01")
PHASES = tu.CUM_METRICS
PARAMS = tu.PARAMS
TRAIN = list(range(1, 71))
VAL = list(range(71, 86))
ALL_DEV = list(range(1, 86))

EXPERIMENTS = [("2003-01-01", 3), ("2003-01-01", 5), ("2004-01-01", 3), ("2005-01-01", 3)]

OBS_WATER = float(curves[(0, "water_rate")].iloc[-1]) if (0, "water_rate") in curves else 7926.36
WATER_CAP = 2.0 * OBS_WATER  # 2x observed 2008 rate

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False})

def save_table(frame, name):
    frame.to_csv(OUT / "tables" / f"{name}.csv", index=False)
    return frame

def show_save(fig, name):
    fig.savefig(OUT / "figures" / f"{name}.png", bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

print(f"Water rate cap: {WATER_CAP:.0f} STB/day ({WATER_CAP/OBS_WATER:.1f}x observed)")
print(f"Dev cases: {len(ALL_DEV)}, Train: {len(TRAIN)}, Val: {len(VAL)}")
print("✅ Ready.")

temporal_utils.py hash: 3ba3b98c
backtest_engine.py hash: 13d59280


Production rows: 5725536
Excluded invalid date/value rows: 5637420


Water rate cap: 15853 STB/day (2.0x observed)
Dev cases: 85, Train: 70, Val: 15
✅ Ready.


In [3]:
# ============================================================
# STRICT B=0 CHALLENGER: Exponential decline + water cap
# ============================================================

def fit_strict_exponential(case, cutoff, window_years=3):
    """Fit strict b=0 exponential to oil rates. Return forecast parameters."""
    cutoff = pd.Timestamp(cutoff)
    history, _ = tu.prefix_history(curves, case, cutoff)
    h = tu.window_frame(history, f"{window_years}y")
    last = history.iloc[-1]
    
    # Oil: strict exponential q(t) = q0 * exp(-D*t)
    t = (h.index - cutoff).total_seconds().to_numpy() / (86400 * YEAR_DAYS)
    qo = float(last["oil_rate"])
    oil_rates = h["oil_rate"].to_numpy(float)
    valid = (oil_rates > 1e-8)
    if valid.sum() < 4:
        raise ValueError(f"Insufficient oil rates: Case {case}")
    
    # Anchored log-decline: D = -sum(t*ln(q/q_cut)) / sum(t^2)
    log_ratio = np.log(oil_rates[valid] / max(qo, 1e-8))
    t_valid = t[valid]
    denom = float(np.dot(t_valid, t_valid))
    D_oil = -float(np.dot(t_valid, log_ratio)) / max(denom, 1e-12)
    D_oil = float(np.clip(D_oil, 0.0, 3.0))  # Physical bound
    
    # Gas: constant GOR from historical median
    gor = float(np.nanmedian(h["gor"].to_numpy(float)))
    if not np.isfinite(gor) or gor <= 0:
        gor = float(last["gas_rate"]) / max(qo, 1e-8)
    
    # Water: relaxation fit q_w(t) = q_w * (ratio + (1-ratio)*exp(-k*t))
    qw = float(last["water_rate"])
    water_rates = h["water_rate"].to_numpy(float)
    water_scale = max(qw, 1.0)
    
    def water_residual(p):
        ratio, k = p
        predicted = qw * (ratio + (1 - ratio) * np.exp(-k * t))
        return (predicted - water_rates) / water_scale
    
    water_fit = least_squares(
        water_residual, x0=[1.2, 0.15],
        bounds=([0.0, 1e-4], [5.0, 3.0]), max_nfev=3000,
    )
    water_ratio, water_k = water_fit.x
    
    return {
        "case_num": int(case), "cutoff": cutoff,
        "anchor_oil": float(last["oil_cum"]),
        "anchor_gas": float(last["gas_cum"]),
        "anchor_water": float(last["water_cum"]),
        "q_oil": qo, "q_water": qw, "q_gas": float(last["gas_rate"]),
        "D_oil": D_oil, "gor": gor,
        "water_ratio": float(water_ratio), "water_k": float(water_k),
    }


def strict_forecast(record, dates, apply_water_cap=True):
    """Generate forecast using strict b=0 exponential + optional water cap."""
    cutoff = pd.Timestamp(record["cutoff"])
    dates = pd.DatetimeIndex(dates)
    if len(dates) == 0 or dates.min() <= cutoff:
        raise ValueError("Dates must be after cutoff.")
    
    # Dense daily grid for accurate integration
    dense = pd.date_range(cutoff, dates.max(), freq="D").union(dates).sort_values()
    elapsed = (dense - cutoff).total_seconds().to_numpy() / 86400  # days
    t = elapsed / YEAR_DAYS  # years
    
    # Oil: strict exponential
    oil_rate = record["q_oil"] * np.exp(-record["D_oil"] * t)
    
    # Gas: constant GOR
    gas_rate = record["gor"] * oil_rate
    
    # Water: relaxation dynamics
    water_rate = record["q_water"] * (
        record["water_ratio"] + (1 - record["water_ratio"]) * np.exp(-record["water_k"] * t)
    )
    
    # Apply water cap
    if apply_water_cap:
        water_rate = np.minimum(water_rate, WATER_CAP)
    
    # Enforce nonnegative rates
    oil_rate = np.maximum(oil_rate, 0.0)
    gas_rate = np.maximum(gas_rate, 0.0)
    water_rate = np.maximum(water_rate, 0.0)
    
    # Integrate cumulative (trapezoid, daily resolution)
    anchors = np.array([record["anchor_oil"], record["anchor_gas"], record["anchor_water"]])
    rates = np.column_stack([oil_rate, gas_rate, water_rate])
    cumulative = anchors + cumulative_trapezoid(rates, elapsed, axis=0, initial=0)
    
    # Extract at requested dates
    positions = dense.get_indexer(dates)
    return cumulative[positions], rates[positions]


# Quick test
test_record = fit_strict_exponential(1, CUTOFF)
test_dates = pd.date_range("2008-04-01", "2028-01-01", freq="QS")
cum, rates = strict_forecast(test_record, test_dates)
print(f"Case 1 test: D_oil={test_record['D_oil']:.4f}, gor={test_record['gor']:.4f}")
print(f"  Oil 2028: {cum[-1,0]:.0f}, Gas: {cum[-1,1]:.0f}, Water: {cum[-1,2]:.0f}")
print(f"  Water capped at {WATER_CAP:.0f} STB/day")
print(f"✅ Strict challenger model working.")

Case 1 test: D_oil=0.1367, gor=0.3634
  Oil 2028: 46733965, Gas: 16975303, Water: 87891822
  Water capped at 15853 STB/day
✅ Strict challenger model working.


In [4]:
# Generate predictions in the format backtest_engine expects
MODEL_ID = "Strict_b0_Anas"

pred_rows = []
fit_records = []

for origin_text, horizon in EXPERIMENTS:
    origin = pd.Timestamp(origin_text)
    forecast_dates = pd.date_range(
        origin + pd.DateOffset(months=3),
        origin + pd.DateOffset(years=horizon),
        freq="QS",
    )
    
    for case in ALL_DEV:
        try:
            record = fit_strict_exponential(case, origin)
            anchor_oil = tu.interpolate_truth(curves[(case, "oil_cum")], [origin])[0]
            anchor_gas = tu.interpolate_truth(curves[(case, "gas_cum")], [origin])[0]
            anchor_water = tu.interpolate_truth(curves[(case, "water_cum")], [origin])[0]
            record["anchor_oil"] = anchor_oil
            record["anchor_gas"] = anchor_gas
            record["anchor_water"] = anchor_water
            
            cum, rates = strict_forecast(record, forecast_dates)
            
            for i, date in enumerate(forecast_dates):
                for j, phase in enumerate(PHASES):
                    pred_rows.append({
                        "model_id": MODEL_ID,
                        "case_num": case,
                        "origin": origin,
                        "horizon_years": horizon,
                        "date": date,
                        "phase": phase,
                        "prediction": cum[i, j],
                    })
            fit_records.append({
                "case_num": case, "origin": origin, "horizon_years": horizon,
                "D_oil": record["D_oil"], "gor": record["gor"],
                "water_ratio": record["water_ratio"], "water_k": record["water_k"],
                "q_oil": record["q_oil"], "q_water": record["q_water"],
            })
        except Exception as e:
            print(f"  SKIP Case {case} @ {origin.year}: {e}")
    
    print(f"  {origin.year}→{horizon}yr: done")

preds = pd.DataFrame(pred_rows)
fits_df = pd.DataFrame(fit_records)
save_table(fits_df, "strict_challenger_fit_parameters")
preds.to_parquet(OUT / "data" / "strict_challenger_backtests.parquet", index=False)

print(f"\nTotal predictions: {len(preds)}")
print(f"Expected: {len(ALL_DEV)} cases × 4 experiments × (12+20+12+12) quarters × 3 phases = {85*56*3}")
print(f"Fit records: {len(fits_df)}")

  2003→3yr: done


  2003→5yr: done


  2004→3yr: done


  2005→3yr: done

Total predictions: 14280
Expected: 85 cases × 4 experiments × (12+20+12+12) quarters × 3 phases = 14280
Fit records: 340


In [5]:
# Score through the standardized referee engine
engine = be.BacktestEngine(curves)

print("=== VALIDATION ===")
errors = engine.validate_submission(preds)
if errors:
    print(f"❌ Validation failed: {errors}")
else:
    print("✅ Submission VALID")

print("\n=== SCORING ===")
report = engine.score_predictions(preds)

if report["validation_errors"]:
    print(f"Errors: {report['validation_errors']}")
else:
    print("\n--- MACRO SCORES ---")
    display(report["macro_scores"])
    
    print("\n--- PHASE SCORES (dev only) ---")
    dev_phases = report["phase_scores"][
        report["phase_scores"]["split"].isin(["train", "validation"])
    ].sort_values(["origin", "horizon_years", "phase"])
    display(dev_phases)
    
    print("\n--- PHYSICAL VIOLATIONS ---")
    display(report["physical_violations"])
    
    print("\n--- VERDICT ---")
    for model, verdict in report["summary"].items():
        print(f"  {model}: {verdict}")
    
    # Export referee report
    engine.export_report(report, OUT / "tables")
    
    # Save quarter scores for trajectory analysis
    report["quarter_scores"].to_csv(
        OUT / "tables" / "strict_challenger_quarter_scores.csv", index=False
    )

=== VALIDATION ===
✅ Submission VALID

=== SCORING ===



--- MACRO SCORES ---


,model_id,mean_dev_NRMSE,worst_exp_NRMSE,median_dev_NRMSE,experiments_scored
0,Strict_b0_Anas,0.152523,0.300618,0.133147,4



--- PHASE SCORES (dev only) ---


,model_id,split,origin,horizon_years,phase,cases,cumulative_RMSE,increment_NRMSE,bias
0,Strict_b0_Anas,train,2003-01-01,3,gas_cum,70,4.725736e+05,0.210504,0.155710
12,Strict_b0_Anas,validation,2003-01-01,3,gas_cum,15,5.066414e+05,0.218749,0.161851
1,Strict_b0_Anas,train,2003-01-01,3,oil_cum,70,1.299794e+06,0.210319,0.155628
13,Strict_b0_Anas,validation,2003-01-01,3,oil_cum,15,1.393066e+06,0.218476,0.161752
2,Strict_b0_Anas,train,2003-01-01,3,water_cum,70,8.227926e+05,0.207155,0.127432
14,Strict_b0_Anas,validation,2003-01-01,3,water_cum,15,7.415291e+05,0.194460,0.111760
3,Strict_b0_Anas,train,2003-01-01,5,gas_cum,70,9.693836e+05,0.301262,0.224403
15,Strict_b0_Anas,validation,2003-01-01,5,gas_cum,15,1.051875e+06,0.316431,0.235714
4,Strict_b0_Anas,train,2003-01-01,5,oil_cum,70,2.666746e+06,0.301048,0.224286
16,Strict_b0_Anas,validation,2003-01-01,5,oil_cum,15,2.893150e+06,0.316125,0.235538



--- PHYSICAL VIOLATIONS ---


,model_id,split,total_predictions,negative_cumulative,negative_cum_pct,decreasing_steps,decreasing_pct
0,Strict_b0_Anas,train,11760,0,0.0,0,0.0
1,Strict_b0_Anas,validation,2520,0,0.0,0,0.0



--- VERDICT ---
  Strict_b0_Anas: PASS | mean_dev=0.1525 | worst_exp=0.3006 | violations=0
Report exported to /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/Anas/outputs/strict_challenger/tables


In [6]:
# Deployable 2028 forecasts for validation cases 71-85
forecast_dates = pd.date_range("2008-04-01", "2028-01-01", freq="QS")
forecast_rows = []

for case in VAL:
    record = fit_strict_exponential(case, CUTOFF)
    cum, rates = strict_forecast(record, forecast_dates)
    
    for i, date in enumerate(forecast_dates):
        forecast_rows.append({
            "case_num": case, "date": date,
            "oil_cum": cum[i, 0], "gas_cum": cum[i, 1], "water_cum": cum[i, 2],
            "oil_rate": rates[i, 0], "gas_rate": rates[i, 1], "water_rate": rates[i, 2],
        })

forecasts_2028 = pd.DataFrame(forecast_rows)
forecasts_2028.to_parquet(OUT / "data" / "strict_challenger_2028.parquet", index=False)
save_table(
    forecasts_2028[forecasts_2028["date"] == pd.Timestamp("2028-01-01")],
    "strict_challenger_2028_endpoints",
)

print(f"2028 forecasts: {len(forecasts_2028)} rows ({len(VAL)} cases × {len(forecast_dates)} dates)")

# Fan chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for j, (phase, unit) in enumerate([
    ("oil_cum", "STB"), ("gas_cum", "MSCF"), ("water_cum", "STB")
]):
    for case in VAL:
        cdata = forecasts_2028[forecasts_2028["case_num"] == case]
        axes[j].plot(cdata["date"], cdata[phase] / 1e6, alpha=0.7)
    axes[j].set(title=f"Strict b=0: {phase}", ylabel=f"Cumulative ({unit}, millions)", xlabel="Date")

fig.suptitle("Strict b=0 Challenger: 2028 Forecasts (Val Cases 71-85)", y=1.02)
fig.tight_layout()
show_save(fig, "strict_challenger_2028_fans")

# Endpoint stats
ep = forecasts_2028[forecasts_2028["date"] == pd.Timestamp("2028-01-01")]
for phase in ["oil_cum", "gas_cum", "water_cum"]:
    print(f"{phase}: mean={ep[phase].mean()/1e6:.1f}M, range=[{ep[phase].min()/1e6:.1f}M, {ep[phase].max()/1e6:.1f}M]")

2028 forecasts: 1200 rows (15 cases × 80 dates)


<Figure size 2160x600 with 3 Axes>

oil_cum: mean=50.6M, range=[39.3M, 60.1M]
gas_cum: mean=18.4M, range=[14.3M, 21.8M]
water_cum: mean=93.7M, range=[73.4M, 112.3M]


In [7]:
# ============================================================
# W4 HANDOFF PACKAGE
# ============================================================

# Manifest
manifest = {
    "workstream": "W4 — Alternative Temporal Structure & Validation",
    "owner": "Anas",
    "model_id": MODEL_ID,
    "temporal_utils_hash": tu_hash,
    "backtest_engine_hash": be_hash,
    "challenger_definition": {
        "oil": "Strict b=0 exponential decline",
        "gas": "Nonnegative constant GOR from historical median",
        "water": "Relaxation dynamics with 2x observed rate cap",
        "integration": "Daily trapezoid from cutoff anchor",
    },
    "experiments": [[o, h] for o, h in EXPERIMENTS],
    "cases": {"train": [1, 70], "validation": [71, 85], "test": [86, 100]},
    "water_cap_STB_per_day": WATER_CAP,
    "outputs": {
        "backtests": "strict_challenger/data/strict_challenger_backtests.parquet",
        "forecasts_2028": "strict_challenger/data/strict_challenger_2028.parquet",
        "referee_report": "strict_challenger/tables/referee_*.csv",
        "fit_params": "strict_challenger/tables/strict_challenger_fit_parameters.csv",
        "endpoints": "strict_challenger/tables/strict_challenger_2028_endpoints.csv",
    },
}

with open(OUT / "w4_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# Handoff markdown
handoff = f"""# W4 Handoff — Strict Timeline & Referee Validation

Owner: Anas | Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}

## Challenger Model: {MODEL_ID}
- Oil: strict b=0 exponential decline (D fitted by anchored log-regression)
- Gas: constant GOR from historical median
- Water: relaxation dynamics capped at {WATER_CAP:.0f} STB/day ({WATER_CAP/OBS_WATER:.1f}x observed 2008)
- Integration: daily trapezoid from cutoff cumulative anchor

## Key Results
- Verdict: {report['summary'].get(MODEL_ID, 'N/A')}
- Backtest predictions: {len(preds)} rows across 4 experiments
- 2028 forecasts: {len(VAL)} validation cases, {len(forecast_dates)} quarterly dates
- Zero physical violations (by construction: nonneg rates + monotonic integration)

## Files
- `backtest_engine.py` — Referee scoring engine (use for ALL team models)
- `strict_challenger/data/strict_challenger_backtests.parquet` — Backtest predictions
- `strict_challenger/data/strict_challenger_2028.parquet` — 2028 forecasts
- `strict_challenger/tables/referee_*.csv` — Scoring report
- `strict_challenger/tables/strict_challenger_fit_parameters.csv` — Fit params per case/origin
- `strict_challenger/w4_manifest.json` — Full configuration

## Prediction Contract
Any team member's predictions must have these columns:
`model_id, case_num, origin, horizon_years, date, phase, prediction`

Phases: oil_cum, gas_cum, water_cum
Origins: 2003-01-01, 2004-01-01, 2005-01-01 (backtests); 2008-01-01 (deployment)
Test cases 86-100: FORBIDDEN in backtests

## For Nazrul (W1), Afiq (W2), Hazeem (W3)
To score your model through the referee engine:
```python
from backtest_engine import BacktestEngine
import temporal_utils as tu
unc, curves = tu.load_shared('path/to/shared/data')
engine = BacktestEngine(curves)
report = engine.score_predictions(your_predictions_df)
print(report['summary'])
```

## Hash Provenance
- temporal_utils.py: {tu_hash}
- backtest_engine.py: {be_hash}
"""

with open(OUT / "W4_HANDOFF.md", "w") as f:
    f.write(handoff)

print("✅ W4 handoff package exported.")
print(f"   Manifest: {OUT / 'w4_manifest.json'}")
print(f"   Handoff:  {OUT / 'W4_HANDOFF.md'}")
print(f"\nAll artifacts in: {OUT}")

✅ W4 handoff package exported.
   Manifest: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/Anas/outputs/strict_challenger/w4_manifest.json
   Handoff:  /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/Anas/outputs/strict_challenger/W4_HANDOFF.md

All artifacts in: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/Anas/outputs/strict_challenger
